# L06 — The Event Calendar and the Simulation Clock

**Module**: M03 | **Chapter**: 4 | **Lecture**: L06

## Learning Objectives
By the end of this notebook you will be able to:
1. Explain the role of the event calendar (future event list) in a DES engine.
2. Implement a minimal event-loop scheduler from scratch in pure Python.
3. Distinguish the simulation clock from wall-clock time.
4. Insert, update, and remove events from a priority-queue calendar.
5. Explain why SimPy's `env.now` is a simulation clock, not a real-time ticker.

---
> **Think → Trace → Code → Experiment → Interpret → Communicate**

We build the DES engine by hand before using SimPy. Understanding this mechanism is what separates practitioners from button-clickers.
---

In [ ]:
import heapq
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

## 1. The Event Calendar Data Structure

The **event calendar** (also called the future event list, or FEL) is a min-heap of `(scheduled_time, event_type, entity_id)` tuples. The simulation engine always processes the *earliest* event first.

Key invariants:
- All entries in the calendar have `scheduled_time >= clock`
- The clock never goes backward
- A single event may schedule zero, one, or more new events

In [ ]:
class EventCalendar:
    """Min-heap event calendar for a hand-coded DES engine."""

    def __init__(self):
        self._heap = []        # (time, tiebreak, event_type, data)
        self._counter = 0      # tiebreak for same-time events

    def schedule(self, time: float, event_type: str, data: dict = None):
        """Insert an event into the calendar."""
        entry = (time, self._counter, event_type, data or {})
        heapq.heappush(self._heap, entry)
        self._counter += 1

    def pop(self):
        """Remove and return the next (earliest) event."""
        time, _, event_type, data = heapq.heappop(self._heap)
        return time, event_type, data

    def peek_time(self) -> float:
        """Return the time of the next event without removing it."""
        return self._heap[0][0] if self._heap else float('inf')

    def is_empty(self) -> bool:
        return len(self._heap) == 0

    def __len__(self):
        return len(self._heap)


# Demonstration: schedule 5 events out of order
cal = EventCalendar()
cal.schedule(5.0,  'departure',  {'customer': 1})
cal.schedule(2.0,  'arrival',    {'customer': 2})
cal.schedule(8.3,  'departure',  {'customer': 2})
cal.schedule(2.0,  'arrival',    {'customer': 3})  # same time as customer 2
cal.schedule(10.0, 'end-of-run', {})

print(f"Calendar has {len(cal)} events. Processing in time order:")
while not cal.is_empty():
    t, etype, data = cal.pop()
    print(f"  t={t:5.1f}  {etype:12s}  {data}")

## 2. The Event Loop — Algorithm 4.1

The DES engine is a **while loop** that: 
1. Pops the earliest event from the calendar
2. Advances the clock to that event's time
3. Executes the event handler (which may schedule new events)
4. Repeats until the calendar is empty or the stop condition is met

In [ ]:
def run_des_engine(initial_events, event_handlers, stop_time: float,
                   state: dict, rng: np.random.Generator):
    """
    Generic DES event loop.

    Parameters
    ----------
    initial_events : list of (time, event_type, data) to seed the calendar
    event_handlers : dict mapping event_type -> callable(clock, state, cal, rng)
    stop_time      : simulation end time
    state          : mutable dict holding all state variables
    rng            : numpy Generator for reproducible randomness

    Returns
    -------
    event_log : list of (time, event_type, state_snapshot) records
    """
    cal = EventCalendar()
    for t, etype, data in initial_events:
        cal.schedule(t, etype, data)

    clock = 0.0
    event_log = []

    while not cal.is_empty() and cal.peek_time() <= stop_time:
        t, etype, data = cal.pop()
        clock = t

        if etype in event_handlers:
            event_handlers[etype](clock, data, state, cal, rng)

        event_log.append({
            'clock': clock,
            'event': etype,
            **{k: v for k, v in state.items() if not k.startswith('_')},
        })

    return event_log

print("run_des_engine defined — ready to use.")

## 3. Apply: Single-Server Queue via Event Loop

We implement an M/M/1 queue using `run_des_engine`. Compare the resulting trace to what SimPy produces for the same seed.

In [ ]:
def build_mm1_handlers(lam: float, mu: float):
    """Return event-handler dict for M/M/1 queue."""

    def on_arrival(clock, data, state, cal, rng):
        cid = data['cid']
        state['arrivals'].append({'cid': cid, 'arrival': clock})

        if state['server_busy']:
            state['queue'].append({'cid': cid, 'join_queue': clock})
        else:
            state['server_busy'] = True
            svc = rng.exponential(1.0 / mu)
            state['_last_svc'][cid] = svc
            cal.schedule(clock + svc, 'departure', {'cid': cid})

        # Schedule next arrival
        next_cid = cid + 1
        ia = rng.exponential(1.0 / lam)
        cal.schedule(clock + ia, 'arrival', {'cid': next_cid})

    def on_departure(clock, data, state, cal, rng):
        cid = data['cid']
        svc = state['_last_svc'].pop(cid, 0.0)
        # Record departure
        for rec in state['arrivals']:
            if rec['cid'] == cid:
                rec['departure'] = clock
                rec['sojourn']   = clock - rec['arrival']
                break

        if state['queue']:
            nxt = state['queue'].pop(0)
            wait = clock - nxt['join_queue']
            nxt_svc = rng.exponential(1.0 / mu)
            state['_last_svc'][nxt['cid']] = nxt_svc
            cal.schedule(clock + nxt_svc, 'departure', {'cid': nxt['cid']})
        else:
            state['server_busy'] = False

    return {'arrival': on_arrival, 'departure': on_departure}


# Run the hand-coded M/M/1 engine
LAM, MU = 3.0, 4.0
RNG = np.random.default_rng(42)
state0 = {'server_busy': False, 'queue': [], 'arrivals': [], '_last_svc': {}}

# Seed first arrival
first_ia = RNG.exponential(1.0 / LAM)
initial = [(first_ia, 'arrival', {'cid': 1})]

log = run_des_engine(initial, build_mm1_handlers(LAM, MU),
                     stop_time=100.0, state=state0, rng=RNG)

log_df = pd.DataFrame(log)
print(f"Events processed: {len(log_df)}")
print(log_df[['clock','event','server_busy']].head(12).to_string(index=False))

In [ ]:
# Compute performance measures from the arrival records
served = [r for r in state0['arrivals'] if 'departure' in r]
sojourns = [r['sojourn'] for r in served]

W_sim = np.mean(sojourns)
W_theory = 1.0 / (MU - LAM)
print(f"Customers served : {len(served)}")
print(f"W  (simulation)  : {W_sim:.4f}")
print(f"W  (M/M/1 theory): {W_theory:.4f}")
print(f"Relative error   : {abs(W_sim - W_theory)/W_theory*100:.2f}%")

## 4. The Clock Visualised

Plot the sequence of clock jumps. Notice: the clock does **not** tick uniformly — it leaps from event to event.

In [ ]:
arrival_times   = log_df[log_df['event']=='arrival']['clock'].values
departure_times = log_df[log_df['event']=='departure']['clock'].values

fig, ax = plt.subplots(figsize=(12, 3))
ax.vlines(arrival_times[:20],   0, 1, colors='steelblue', lw=1.2, label='Arrival events')
ax.vlines(departure_times[:20], 0, 1, colors='tomato',    lw=1.2, label='Departure events')
ax.set_xlim(0, departure_times[19] * 1.05)
ax.set_xlabel('Simulation clock (virtual time)')
ax.set_yticks([])
ax.legend()
ax.set_title('Clock jumps: the simulation clock leaps from event to event (first 20 of each)')
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

gaps = np.diff(sorted(np.concatenate([arrival_times[:20], departure_times[:20]])))
print(f"Mean gap between consecutive events (first 40): {gaps.mean():.4f}")
print(f"Min gap: {gaps.min():.4f}   Max gap: {gaps.max():.4f}")
print("The clock can sit idle (no events) for long stretches — no work is done in between.")

## 5. SimPy Uses the Same Engine

SimPy's `Environment` is exactly this event-calendar engine wrapped in Python generators. Let's verify that `env.now` is the simulation clock — it jumps to the next scheduled event, not to a real-time tick.

In [ ]:
import simpy

clock_snapshots = []

def clock_watcher(env):
    """Process that records clock time at every wakeup."""
    for _ in range(10):
        # Request a timeout drawn from Exp(1)
        wait = np.random.exponential(1.0)
        yield env.timeout(wait)
        clock_snapshots.append(env.now)

env = simpy.Environment()
env.process(clock_watcher(env))
env.run()

gaps_simpy = [clock_snapshots[0]] + list(np.diff(clock_snapshots))
print("SimPy env.now at each wakeup (clock jumps):")
for i, (t, g) in enumerate(zip(clock_snapshots, gaps_simpy), 1):
    print(f"  wakeup {i:2d}: env.now = {t:7.4f}  (gap = {g:.4f})")

print()
print("Gaps are exponential random variables — the clock is NOT uniform.")

## 6. Three-Phase Execution (A–B–C)

Some textbooks describe DES using the **three-phase** approach instead of event-scheduling:

| Phase | Action |
|---|---|
| **A** | Advance clock to the next event time |
| **B** | Execute all **bound** events (unconditional — time has come) |
| **C** | Execute all **conditional** events (state condition now satisfied) |

SimPy uses the event-scheduling worldview (not three-phase), but the two are equivalent.

In [ ]:
# Trace a 3-phase execution for the first 5 events in our M/M/1 run
# to see how A-B-C maps onto the event log we already produced

sample = log_df.head(8)
print("Three-phase interpretation of our event log:")
print(f"{'Phase A: advance clock':30s}  {'Phase B: bound event':20s}  Phase C: any freed resources?")
print('-' * 85)

prev_clock = 0.0
for _, row in sample.iterrows():
    phase_A = f"0 → {row['clock']:.3f}"
    phase_B = f"{row['event']}"
    phase_C = "serve next customer" if row['event']=='departure' and not row['server_busy'] else "—"
    print(f"  clock {phase_A:22s}  {phase_B:22s}  {phase_C}")
    prev_clock = row['clock']

---
## Try It Yourself

1. **Priority tiebreaking**: In the event calendar, two events at the same time must be ordered. Modify `EventCalendar.schedule` to accept a `priority` argument (lower = earlier). Re-run the M/M/1 to always process departures before arrivals at the same clock tick. Does the steady-state W change?

2. **Event cancellation**: Some systems need to cancel events (e.g., a customer leaves before being served — reneging). Add a `cancel(event_id)` method to `EventCalendar` using a "lazy deletion" scheme (mark cancelled events; skip them on `pop`). Test by simulating a customer who reneges after waiting 5 minutes.

3. **SimPy mapping**: Open [simdes/simdes/core/model.py](../../../simdes/simdes/core/model.py) and find the `run()` method. Explain in one paragraph how SimPy's `env.run(until=...)` maps onto the `while not cal.is_empty()` loop you wrote in this notebook.